# Midas Design Guide — Steel Composite Girder Design

**Companion notebook** for the corresponding chapter of the MIDAS training
manual *Design Guide for midas Civil — AASHTO LRFD*. The guide itself is
proprietary and is **not reproduced here** — this notebook contains only
original code and AASHTO LRFD / MBE article citations.

**How this notebook is used.** Work through the steel-composite design
process with this notebook alongside: each step carries its AASHTO
background in original words, a live Python environment for exploratory
checks and quick validation, a direct interface to Midas Civil through its
API, and customization to ODOT practice (Grade 50W unpainted weathering
steel, ODOT deck and haunch conventions, BDM load assumptions).
The guide's steel chapter, unlike its PSC chapter, contains **no worked
numerical example** — it presents the check equations and the Midas dialog
fields that drive them. So this notebook supplies the missing example: a
realistic ODOT-style two-span composite plate girder, carried through
every limit state with numbers a designer can rerun and modify. Where a
number must come from the Midas model itself (staged analysis forces,
envelope results, design result tables), there is a `TODO(midas-api)`
marker — those cells get finished against the live Civil NX JSON API when
a steel composite model exists.

**Scope of this chapter:**

1. The demo girder — an ODOT-style two-span composite plate girder
2. Cross-section proportion limits (LRFD 6.10.2)
3. Plastic moment $M_p$ and yield moment $M_y$ (Appendix D6), section
   classification (6.10.6.2), and the $D_p \le 0.42D_t$ ductility rule
4. Composite section properties — bare steel / $n$ / $3n$ / cracked, and
   effective flange width (6.10.1.1, 4.6.2.6)
5. Distribution factors and force effects (4.6.2.2; Midas API)
6. Constructibility — the noncomposite girder under wet concrete (6.10.3)
7. Service II permanent-deformation limits (6.10.4)
8. Fatigue (6.6.1, 6.10.5)
9. Strength I flexure — compact positive (6.10.7), noncompact negative
   (6.10.8), and the Appendix A6 path
10. Web shear and tension-field action (6.10.9)
11. Shear connectors — fatigue pitch and strength totals (6.10.10)
12. Transverse and bearing stiffeners (6.10.11)
13. Midas design-variable mapping and design result tables
14. What changed between editions — the guide's 4th-vs-6th list, extended
    to the current 9th

**What the `check()`s compare.** Unless a cell says otherwise, a `check()`
validates one Python derivation against another — the cell's hand calc vs
civilpy's implementation — or against a published constant (an AASHTO
limit, an AISC/ODOT table value). That proves the walkthrough and the
library agree on the article's math; it is *not* a software cross-check.
Labels mentioning Midas mean one of two things: a **Midas convention
computed here** (e.g. the D6.1 PNA-case table or the $M_y$ staged-modulus
iteration exactly as the guide documents them, evaluated in Python for
comparison), or a **Midas model value** pulled from Civil NX over the API
— live when connected, otherwise recorded values noted in the cell.
Midas's *design-module* result tables are not API-reachable, so
civilpy-vs-Midas design-check comparisons remain manual side-by-sides.

**Guide editions.** This notebook was written against the **2020 edition**
of the guide; the PDF archived in `snbi_ui/utils/manuals/Midas/` (from the
Midas support KB) is the **2014 edition**, which teaches this chapter to
AASHTO LRFD **4th (2007) and 6th (2012)** side by side. Unlike the
concrete articles (renumbered wholesale in 2017), the steel articles have
kept their numbers: 6.10.x (I-sections), 6.11.x (box/tub), and the
Appendix A6 / B6 / D6 machinery cited here read the same in the 4th
edition and the current **9th (2020)**. What changed between editions is
mostly *content* inside stable article numbers — the guide's own
"Difference Between AASHTO-LRFD 4th and 6th" section lists the 2007→2012
deltas, and the closing section of this notebook extends that list
through the 9th. Known guide-vs-current disagreements are flagged in
place with ⚠️ callouts.

In [ ]:
import math
import pandas as pd

# civilpy is installed editable into this env (pip install -e .)
from civilpy.structural.midas import MidasCivil, parse_result_table, envelope
from civilpy.structural.aashto.lrfd import (
    concrete, prestressed, steel, composite, distribution, lrfr, creep_shrinkage,
)

# --- Midas Civil NX connection -------------------------------------------------
# The API is not running on this machine right now. Everything below that needs
# the live model is guarded by MIDAS_ONLINE and marked TODO(midas-api).
try:
    midas = MidasCivil()
    MIDAS_ONLINE = midas.ping()
except Exception:
    midas, MIDAS_ONLINE = None, False
print("Midas Civil NX online:", MIDAS_ONLINE)

In [ ]:
# --- Validation harness --------------------------------------------------------
# Every comparison in this notebook goes through check() so the end-of-notebook
# summary shows guide value vs civilpy value side by side.
RESULTS = []

def check(label, guide_value, civilpy_value, tol=0.01, unit=""):
    """Compare a guide-reported value against the civilpy-computed one.

    tol is relative (1% default) — the guide rounds intermediate values, so
    small drift is expected; flag anything beyond tol for investigation.
    """
    if guide_value is None or civilpy_value is None:
        status = "PENDING"
        diff = None
    else:
        diff = abs(civilpy_value - guide_value) / (abs(guide_value) or 1.0)
        status = "OK" if diff <= tol else "MISMATCH"
    RESULTS.append({"check": label, "guide": guide_value, "civilpy": civilpy_value,
                    "rel diff": diff, "unit": unit, "status": status})
    print(f"[{status}] {label}: guide={guide_value} civilpy={civilpy_value} {unit}")
    return status == "OK"

def summary():
    df = pd.DataFrame(RESULTS)
    if len(df):
        n_ok = (df.status == "OK").sum()
        print(f"{n_ok}/{len(df)} checks OK, "
              f"{(df.status == 'MISMATCH').sum()} mismatches, "
              f"{(df.status == 'PENDING').sum()} pending")
    return df

## 1. The demo girder — an ODOT-style two-span plate girder

The guide presents this chapter's checks abstractly, so the notebook runs
them on a concrete example sized the way an Ohio designer would:

| Item | Value | ODOT practice note |
|---|---|---|
| Spans | 2 × 120 ft continuous | two-span exercises *both* flexure signs |
| Girders | 4 @ 9'-0" spacing | interior girder designed here |
| Deck | 8.5 in structural | ODOT standard composite deck, $f'_c$ = 4.5 ksi (Class QC2) |
| Haunch | 2 in (top of top flange → bottom of deck) | ODOT convention; concrete over the flange only |
| Steel | Grade 50W, $F_y$ = 50 ksi, $F_u$ = 70 ksi | unpainted weathering steel — ODOT's default for new girders |
| Field section | TF 16×1, web 54×½, BF 18×1⅜ | positive-flexure region |
| Pier section | TF 18×1½, web 54×⁹⁄₁₆, BF 20×1¾ | negative-flexure region |

Why these plates: a 54-in web puts the steel depth near $0.033L$ —
comfortably above the $0.027L$ steel-depth minimum of Table 2.5.2.6.3-1
for continuous composite I-girders — and a two-plate schedule (field +
pier) is the smallest realistic set that shows how the checks change sign.
The bottom flange is wider and thicker than the top in *both* regions:
in positive flexure the deck does the compression work so the top flange
mainly needs constructibility stiffness; in negative flexure the bottom
flange is the discretely braced compression flange and earns its area.

The two cross sections are also mirrored as `GirderSide` objects — the
same dataclass civilpy's bolted-field-splice and composite-section
machinery consume — so every later section can pull geometry from one
place.

In [ ]:
# The single source of truth for the demo bridge. Everything downstream
# reads from DEMO / FIELD / PIER — change a plate here and rerun.
from civilpy.structural.aashto.lrfd.bolted_field_splice import Flange, GirderSide

DEMO = dict(
    spans_ft=[120.0, 120.0],       # two-span continuous
    girder_spacing_ft=9.0,
    n_girders=4,
    deck_thickness_in=8.5,         # structural thickness
    haunch_in=2.0,                 # top of top flange -> bottom of deck
    fc_deck_ksi=4.5,               # ODOT Class QC2
    fy_ksi=50.0,                   # Grade 50W
    fu_ksi=70.0,
    unit_weight_conc_kcf=0.150,
)

# plate schedule: (b_tf x t_tf, D x t_w, b_bf x t_bf), inches
FIELD = GirderSide(
    top_flange=Flange("Grade 50W", thickness=1.0, width=16.0),
    bottom_flange=Flange("Grade 50W", thickness=1.375, width=18.0),
    web_material="Grade 50W", web_thickness=0.5, web_depth=54.0,
    haunch=DEMO["haunch_in"],
)
PIER = GirderSide(
    top_flange=Flange("Grade 50W", thickness=1.5, width=18.0),
    bottom_flange=Flange("Grade 50W", thickness=1.75, width=20.0),
    web_material="Grade 50W", web_thickness=0.5625, web_depth=54.0,
    haunch=DEMO["haunch_in"],
)

def steel_summary(side, label):
    a = (side.top_flange.area + side.bottom_flange.area
         + side.web_thickness * side.web_depth)
    d = side.top_flange.thickness + side.web_depth + side.bottom_flange.thickness
    return dict(section=label, A_steel_in2=a, depth_in=d,
                weight_plf=round(a * 3.4, 1))

df_sections = pd.DataFrame([steel_summary(FIELD, "field (positive)"),
                            steel_summary(PIER, "pier (negative)")])

# steel depth vs the Table 2.5.2.6.3-1 minimum for continuous composite
# I-girders (0.027L) -- a proportioning sanity check, not a strength check
L = DEMO["spans_ft"][0]
d_steel = df_sections.depth_in.min()
print(f"steel depth {d_steel:.2f} in vs 0.027L = {0.027 * L * 12:.2f} in minimum")
check("steel depth >= 0.027L (Table 2.5.2.6.3-1) [published limit]",
      1.0, float(d_steel >= 0.027 * L * 12.0))
df_sections

## 2. Cross-section proportion limits — LRFD 6.10.2

Before any strength math, 6.10.2 screens the raw plate proportions. These
limits are not strength checks — they are the *applicability boundary* of
everything that follows: the flexure and shear resistances of 6.10.7–6.10.9
were calibrated on girders inside these ratios, and Midas reports a plain
"NG" when a section steps outside them. What each limit is protecting:

- **Web slenderness** $D/t_w \le 150$ without longitudinal stiffeners
  (6.10.2.1.1-1), $\le 300$ with them (6.10.2.1.2-1). Keeps the panel
  handleable in the shop and in lifting, and keeps elastic web
  bend-buckling from governing so early that the section is uneconomical.
- **Flange slenderness** $b_f/2t_f \le 12.0$ (6.10.2.2-1). A stockiness
  cap so the flange can be welded, gripped, and erected without folding,
  and so flange local buckling stays in the range the 6.10.8.2.2 curve
  covers.
- **Flange width** $b_f \ge D/6$ (6.10.2.2-2). The lateral-torsional
  buckling equations model the girder as flanges restraining a web; a
  too-narrow flange breaks that model.
- **Flange thickness** $t_f \ge 1.1t_w$ (6.10.2.2-3). The web
  bend-buckling coefficient assumes the flanges act as rotational
  restraints at the web's edges — the flange must be meaningfully stiffer
  than the plate it is bracing.
- **Flange inertia ratio** $0.1 \le I_{yc}/I_{yt} \le 10$ (6.10.2.2-4).
  Outside this band the section behaves like a tee, and the LTB
  derivations (which assume something I-shaped) stop applying.

> ⚠️ **Guide-vs-current note.** The guide presents these tables for the
> 4th/6th editions; the limits are unchanged in the 9th (2020) — same
> numbers, same article layout — so its table transfers cleanly. For
> box/tub sections the parallel limits live in 6.11.2 (webs may be
> inclined; the web limit applies along the slope).

A practical companion rule worth knowing at this stage: C6.10.3.4.1
suggests $b_{fc} \ge L_{ship}/85$ for the *shipped field piece* so girders
survive handling before the deck exists — with 120-ft spans and a field
splice near the 0.7 point, a ~90-ft piece wants a compression flange of
at least ~12.7 in. Both demo flanges clear it.

In [ ]:
# Hand-calc every 6.10.2 ratio, then confirm civilpy's verdicts match.
rows = []
for label, side in (("field", FIELD), ("pier", PIER)):
    D, tw = side.web_depth, side.web_thickness
    for fl_label, fl in (("top", side.top_flange), ("bottom", side.bottom_flange)):
        rows.append(dict(section=label, flange=fl_label,
                         ratio="bf/2tf", value=fl.width / (2 * fl.thickness),
                         limit="<= 12", ok=fl.width / (2 * fl.thickness) <= 12))
        rows.append(dict(section=label, flange=fl_label,
                         ratio="bf >= D/6", value=fl.width,
                         limit=f">= {D/6:.1f}", ok=fl.width >= D / 6))
        rows.append(dict(section=label, flange=fl_label,
                         ratio="tf >= 1.1tw", value=fl.thickness,
                         limit=f">= {1.1*tw:.3f}", ok=fl.thickness >= 1.1 * tw))
    rows.append(dict(section=label, flange="-", ratio="D/tw",
                     value=D / tw, limit="<= 150", ok=D / tw <= 150))
    i_yt = side.bottom_flange.thickness * side.bottom_flange.width ** 3 / 12
    i_yc = side.top_flange.thickness * side.top_flange.width ** 3 / 12
    rows.append(dict(section=label, flange="-", ratio="Iyc/Iyt",
                     value=i_yc / i_yt, limit="0.1 - 10",
                     ok=0.1 <= i_yc / i_yt <= 10))
df_prop = pd.DataFrame(rows)
display(df_prop)

# civilpy's one-call screen per region (positive flexure: top flange is
# the compression flange; the function is orientation-agnostic for these
# geometric limits) -- hand calc vs civilpy
for label, side in (("field", FIELD), ("pier", PIER)):
    i_yc = side.top_flange.thickness * side.top_flange.width ** 3 / 12
    i_yt = side.bottom_flange.thickness * side.bottom_flange.width ** 3 / 12
    res = steel.proportion_limits(
        d_web=side.web_depth, t_w=side.web_thickness,
        b_fc=side.top_flange.width, t_fc=side.top_flange.thickness,
        b_ft=side.bottom_flange.width, t_ft=side.bottom_flange.thickness,
        i_yc=i_yc, i_yt=i_yt)
    hand_ok = df_prop[df_prop.section == label].ok.all()
    check(f"6.10.2 all proportion limits, {label} section [hand calc]",
          float(hand_ok), res.capacity)

# shipping-piece flange guideline, C6.10.3.4.1 [published rule of thumb]
L_ship_ft = 0.75 * DEMO["spans_ft"][0]     # field piece to the splice
b_min_ship = L_ship_ft * 12 / 85
print(f"shipped-piece minimum flange: {b_min_ship:.1f} in "
      f"(field top flange = {FIELD.top_flange.width} in)")
check("C6.10.3.4.1 bfc >= Lship/85, field top flange [published value]",
      1.0, float(FIELD.top_flange.width >= b_min_ship))

## 3. Composite section properties

Short-term (n), long-term (3n), and cracked-section properties per region;
effective flange width per 4.6.2.6.

In [ ]:
# TODO(guide):
# n = composite.modular_ratio(fc=...)
# girder = composite.CompositeGirder(...)  # per plate region
# check() section moduli against the guide's tables.
pass

## 4. Distribution factors and force effects

DFs per 4.6.2.2 (steel I-girder row), then the unfactored envelopes from the
Midas model — DC1/DC2/DW split matters for the 3n vs n stress buildup.

In [ ]:
# TODO(guide): distribution.moment_df_interior(...) etc., as in the PSC
# chapter, then:

In [ ]:
if MIDAS_ONLINE:
    # Verified vs live Civil NX 2026-07-27: /post/TABLE selects by
    # TABLE_TYPE ("BEAMFORCE"); TABLE_NAME is just a label.
    try:
        resp = midas.result_table("BeamForce",
                                  table_type="BEAMFORCE")
        display(pd.DataFrame(parse_result_table(resp)).head())
    except Exception as err:
        # a fresh/unanalyzed session (or a DB edit, or a pre-mode view
        # switch) clears results — analyze and rerun this cell
        print("no results in the session:", str(err)[-80:])
else:
    print("Midas offline — skipping BeamForce (BEAMFORCE)")

## 5. Constructibility — LRFD 6.10.3

Deck-casting sequence stresses on the noncomposite section: flange nominal
yielding, flange local buckling, LTB with the casting unbraced lengths, and
web bend-buckling.

In [ ]:
# TODO(guide):
# steel.constructibility_compression_flange(...)
# steel.web_bend_buckling(...)
# steel.lateral_torsional_buckling_resistance(...)
# TODO(midas-api): construction-stage results table for the casting sequence.
pass

## 6. Service II — LRFD 6.10.4

Flange stress limits 0.95·Rh·Fyf (composite) / 0.80·Rh·Fyf, with the
1.3·LL+IM Service II combination.

In [ ]:
# TODO(guide): build f_f from the staged section moduli (DC1 on steel,
# DC2+DW on 3n, LL on n); steel.hybrid_factor(...) if hybrid.
pass

## 7. Fatigue — LRFD 6.10.5 / 6.6.1

Governing details (web-to-flange weld, stiffener welds, shear studs), fatigue
category resistance, and the Fatigue I/II stress ranges from the fatigue
truck.

In [ ]:
# TODO(guide): steel.fatigue_resistance(category=..., n_cycles=...)
# TODO(midas-api): fatigue truck moving-load stress ranges from the model.
pass

## 8. Strength I flexure

Positive flexure: compact composite section check and Dp/Dt ductility
(6.10.7). Negative flexure: FLB / LTB per 6.10.8, or Appendix A6 if the guide
uses it for the compact-web continuous section.

In [ ]:
# TODO(guide):
# steel.compact_composite_positive_flexure(...)
# steel.flange_local_buckling_resistance(...)
# steel.lateral_torsional_buckling_resistance(...)
# steel.tension_flange_resistance(...)
# civilpy.structural.aashto.lrfd.appendix_a6 for the A6 path if used.
pass

## 9. Shear — LRFD 6.10.9

End/interior panel shear with tension-field action, transverse stiffener
spacing, stiffener proportioning (6.10.11.1).

In [ ]:
# TODO(guide):
# steel.web_shear_resistance(...)
# steel.transverse_stiffener_width(...), steel.transverse_stiffener_inertia(...)
pass

## 10. Shear connectors — LRFD 6.10.10

Fatigue pitch governing the layout, strength check of total connectors between
points of max moment and zero moment.

In [ ]:
# TODO(guide):
# steel.shear_connector_strength(...)
# steel.shear_connector_fatigue_pitch(...)
pass

## 11. Bearing stiffeners and Midas cross-check

In [ ]:
# TODO(guide): steel.bearing_stiffener_resistance(...), bearing_stiffener_width(...)

In [ ]:
# TODO(midas-api): "Composite Girder Design Result" is a PSC *design-module* table.
# Verified 2026-07-27: the design result tables (fps, c, Mcr, Av,req,
# FDL/AFDL columns) are NOT exposed through the known /post/TABLE surface —
# they need the PSC Design run configured in the Civil NX UI (design code,
# PSC design parameters, Section Manager rebar) and/or the official JSON
# manual's design TABLE_TYPE names. Analysis-side tables ARE verified:
# BEAMFORCE, BEAMSTRESSPSC (the ten-check-point stress table with
# Sig-Is(shear), Sig-Is(shear+torsion), Sig-Ps(Max/Min) columns), REACTIONG.
print("PENDING: Composite Girder Design Result — needs the PSC Design module (see comment)")

## Validation summary

Every `check()` recorded above, in one table. `PENDING` rows are waiting on
either guide values (hand entry) or the Midas API coming back online.

In [ ]:
summary()